# Buổi 5: Xử lý và biến đổi dữ liệu với Pandas
**Môn học:** Python cho Phân tích Dữ liệu Kinh doanh · **BÀI TẬP THỰC HÀNH**

**Mục tiêu:**
- Phát hiện và xử lý dữ liệu trùng lặp, thiếu, sai định dạng, ngoại lai
- Chuyển kiểu dữ liệu, xử lý ngày tháng, tạo cột mới
- Tổng hợp bằng groupby, pivot_table; kết hợp bảng bằng merge, concat
- Tính chỉ số RFM cho khách hàng

**Dữ liệu dùng trong buổi:** `ecommerce_orders_raw.csv`, `customers.csv` (trong thư mục `data/`). Nếu dùng Google Colab, tải thư mục `data/` lên rồi sửa biến `DATA` bên dưới.

**Cấu trúc:** Phần A – bài tập có hướng dẫn (làm theo, chạy từng ô) · Phần B – bài tập tự làm (bắt buộc, nộp cuối buổi) · Phần C – bài tập làm thêm (dành cho bạn muốn thử thách).

In [1]:
import numpy as np
import pandas as pd
DATA = "../data/"
pd.set_option("display.max_columns", 30)

## Phần A. Bài tập có hướng dẫn

Làm sạch bộ dữ liệu đơn hàng thương mại điện tử năm 2025 rồi phân tích.

### A1. Khám phá vấn đề của dữ liệu thô

In [2]:
raw = pd.read_csv(DATA + "ecommerce_orders_raw.csv")
print(raw.shape)
raw.info()
raw.sample(8, random_state=3)

(3816, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3816 entries, 0 to 3815
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ma_don      3816 non-null   object 
 1   ngay_dat    3816 non-null   object 
 2   ma_kh       3816 non-null   object 
 3   vung        3796 non-null   object 
 4   kenh        3816 non-null   object 
 5   nhom_hang   3816 non-null   object 
 6   san_pham    3816 non-null   object 
 7   so_luong    3755 non-null   float64
 8   don_gia     3816 non-null   object 
 9   trang_thai  3816 non-null   object 
dtypes: float64(1), object(9)
memory usage: 298.2+ KB


,ma_don,ngay_dat,ma_kh,vung,kenh,nhom_hang,san_pham,so_luong,don_gia,trang_thai
3087,DH001670,2025-05-22,KH0261,Miền Bắc,Website,Mỹ phẩm,Mỹ phẩm C,1.0,498000,Hoàn thành
454,DH002270,2025-02-13,KH0348,Miền Bắc,Sàn TMĐT,Mỹ phẩm,Mỹ phẩm C,1.0,461000,Hoàn thành
3255,DH003292,2025-07-24,KH0525,Miền Bắc,Cửa hàng,Điện thoại,Điện thoại C,2.0,4611000,Hoàn thành
1417,DH000787,2025-04-16,KH0122,Miền Nam,Sàn TMĐT,Phụ kiện,Phụ kiện D,1.0,493000,Đã hủy
6,DH000271,2025-09-06,KH0045,Miền Nam,App,Điện thoại,Điện thoại D,1.0,4919000,Hoàn thành
3397,DH002100,2025-06-13,KH0322,Miền Nam,Sàn TMĐT,Mỹ phẩm,Mỹ phẩm B,NaN,424000,Hoàn thành
2831,DH000274,17/06/2025,KH0045,Miền Nam,App,Thời trang,Thời trang B,3.0,386000,Hoàn thành
2340,DH002347,2025-07-06,KH0360,Miền Nam,App,Điện thoại,Điện thoại E,1.0,7333000,Hoàn thành


In [3]:
print("Giá trị thiếu mỗi cột:\n", raw.isna().sum())
print("\nSố dòng trùng hoàn toàn:", raw.duplicated().sum())
print("\nCác giá trị cột vùng:", raw["vung"].unique())
print("\nVí dụ đơn giá:", raw["don_gia"].sample(6, random_state=1).tolist())

Giá trị thiếu mỗi cột:
 ma_don         0
ngay_dat       0
ma_kh          0
vung          20
kenh           0
nhom_hang      0
san_pham       0
so_luong      61
don_gia        0
trang_thai     0
dtype: int64

Số dòng trùng hoàn toàn: 45

Các giá trị cột vùng: ['Miền Nam' 'Miền Trung' 'Miền Bắc' nan ' Miền Nam ' ' Miền Bắc '
 ' Miền Trung ' 'MIỀN NAM' 'MIỀN TRUNG' 'MIỀN BẮC']

Ví dụ đơn giá: ['467000', '287000', '472000', '9109000', '8070000', '263000']


**Nhận xét:** (1) có dòng trùng; (2) `vung` viết không thống nhất và có giá trị thiếu; (3) `so_luong` thiếu;
(4) `don_gia` là chuỗi, có dạng "1.250.000 đ", có giá trị âm và giá trị bất thường; (5) `ngay_dat` lẫn hai định dạng.

### A2. Bỏ trùng lặp

In [4]:
df = raw.drop_duplicates().copy()
print("Trước:", len(raw), "→ Sau:", len(df))
print("Mã đơn còn trùng?", df["ma_don"].duplicated().sum())

Trước: 3816 → Sau: 3771
Mã đơn còn trùng? 0


### A3. Chuẩn hóa chuỗi và kiểu số

In [5]:
df["vung"] = df["vung"].str.strip().str.title()
print(df["vung"].value_counts(dropna=False))

def lam_sach_gia(x):
    if isinstance(x, str):
        x = x.replace("đ", "").replace(".", "").strip()
    return float(x)

df["don_gia"] = df["don_gia"].apply(lam_sach_gia)
df["so_luong"] = pd.to_numeric(df["so_luong"], errors="coerce")
df[["don_gia", "so_luong"]].describe()

vung
Miền Nam      1633
Miền Bắc      1344
Miền Trung     774
NaN             20
Name: count, dtype: int64


,don_gia,so_luong
count,3.771000e+03,3711.000000
mean,4.889065e+06,1.683374
std,3.668363e+07,0.745691
min,-1.000000e+00,1.000000
25%,3.830000e+05,1.000000
50%,5.400000e+05,2.000000
75%,5.633000e+06,2.000000
max,9.999990e+08,3.000000


### A4. Ngày tháng
`pd.to_datetime` với `format="mixed"` và `dayfirst=True` hiểu được cả `2025-03-15` lẫn `15/03/2025`.
Sau đó trích xuất tháng, quý, thứ trong tuần bằng `.dt`.

In [6]:
df["ngay_dat"] = pd.to_datetime(df["ngay_dat"], format="mixed", dayfirst=True)
df["thang"] = df["ngay_dat"].dt.month
df["quy"] = df["ngay_dat"].dt.quarter
df["thu"] = df["ngay_dat"].dt.day_name()
print(df["ngay_dat"].min(), "→", df["ngay_dat"].max())
df[["ngay_dat", "thang", "quy", "thu"]].head()

2025-01-01 00:00:00 → 2025-12-31 00:00:00


,ngay_dat,thang,quy,thu
0,2025-01-24,1,1,Friday
1,2025-12-18,12,4,Thursday
2,2025-06-05,6,2,Thursday
3,2025-03-06,3,1,Thursday
4,2025-01-26,1,1,Sunday


### A5. Giá trị thiếu và ngoại lai
Chiến lược phổ biến: **xóa** dòng (khi ít và không quan trọng), **điền** giá trị (trung vị, mode, giá trị từ nguồn khác),
hoặc **giữ** và đánh dấu. Ngoại lai phát hiện bằng quy tắc IQR: ngoài khoảng `[Q1 − 1,5·IQR; Q3 + 1,5·IQR]`
tính **trong từng nhóm hàng** (vì laptop vốn đắt hơn phụ kiện).

In [7]:
# 1) số lượng thiếu → điền bằng trung vị (1)
df["so_luong"] = df["so_luong"].fillna(df["so_luong"].median())

# 2) vùng thiếu → lấy từ bảng khách hàng
kh = pd.read_csv(DATA + "customers.csv")
vung_kh = kh.set_index("ma_kh")["vung"]
df["vung"] = df["vung"].fillna(df["ma_kh"].map(vung_kh))

# 3) đơn giá âm → lỗi nhập liệu, loại bỏ
df = df[df["don_gia"] > 0]

# 4) ngoại lai theo IQR trong từng nhóm hàng
def la_ngoai_lai(g):
    q1, q3 = g.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (g < q1 - 1.5 * iqr) | (g > q3 + 1.5 * iqr)

df["ngoai_lai"] = df.groupby("nhom_hang")["don_gia"].transform(la_ngoai_lai)
print("Số ngoại lai:", df["ngoai_lai"].sum())
print(df[df["ngoai_lai"]][["ma_don", "nhom_hang", "don_gia"]])
df = df[~df["ngoai_lai"]].drop(columns="ngoai_lai")
print("Còn lại:", len(df), "dòng; thiếu:", df.isna().sum().sum())

Số ngoại lai: 5
        ma_don   nhom_hang      don_gia
789   DH003663  Thời trang  999999000.0
863   DH001732    Phụ kiện  999999000.0
1635  DH003565  Điện thoại  999999000.0
2503  DH003296     Mỹ phẩm  999999000.0
3731  DH000032  Thời trang  999999000.0
Còn lại: 3761 dòng; thiếu: 0


### A6. Cột mới, lọc đơn hoàn thành và tổng hợp

In [8]:
df["so_luong"] = df["so_luong"].astype(int)
df["doanh_thu"] = df["so_luong"] * df["don_gia"]
ht = df[df["trang_thai"] == "Hoàn thành"].copy()
print("Tỷ lệ hủy đơn:", round((df["trang_thai"] == "Đã hủy").mean() * 100, 1), "%")

theo_thang = ht.groupby("thang").agg(so_don=("ma_don", "count"),
                                     doanh_thu_ty=("doanh_thu", lambda x: x.sum() / 1e9))
theo_thang.round(2)

Tỷ lệ hủy đơn: 19.8 %


,so_don,doanh_thu_ty
thang,,
1,429,2.15
2,206,1.18
3,234,1.43
4,230,1.76
5,214,1.26
6,217,1.46
7,217,1.15
8,212,1.18
9,203,1.38


`pivot_table` – bảng tổng hợp hai chiều giống PivotTable trong Excel:

In [9]:
pv = pd.pivot_table(ht, index="nhom_hang", columns="kenh", values="doanh_thu",
                    aggfunc="sum", margins=True, margins_name="Tổng") / 1e6
pv.round(0)

kenh,App,Cửa hàng,Sàn TMĐT,Website,Tổng
nhom_hang,,,,,
Gia dụng,183.0,90.0,156.0,170.0,599.0
Laptop,3206.0,1676.0,3100.0,2637.0,10619.0
Mỹ phẩm,107.0,46.0,101.0,77.0,330.0
Phụ kiện,140.0,66.0,142.0,122.0,470.0
Thời trang,134.0,60.0,147.0,125.0,466.0
Điện thoại,1438.0,840.0,1550.0,1606.0,5433.0
Tổng,5207.0,2779.0,5195.0,4736.0,17917.0


`merge` ghép thông tin khách hàng (tuổi, giới tính) vào đơn hàng:

In [10]:
ht = ht.merge(kh[["ma_kh", "tuoi", "gioi_tinh"]], on="ma_kh", how="left")
ht["nhom_tuoi"] = pd.cut(ht["tuoi"], bins=[17, 25, 35, 50, 65], labels=["18-25", "26-35", "36-50", "51-65"])
ht.groupby("nhom_tuoi", observed=True)["doanh_thu"].agg(["count", "mean"]).round(0)

,count,mean
nhom_tuoi,,
18-25,456,6433509.0
26-35,705,6150372.0
36-50,902,5978629.0
51-65,953,5513939.0


### A7. Phân tích RFM
RFM đánh giá giá trị khách hàng bằng 3 chỉ số: **R**ecency – số ngày từ lần mua gần nhất,
**F**requency – số đơn, **M**onetary – tổng chi tiêu. Mỗi chỉ số chấm điểm 1–5 bằng ngũ phân vị (`pd.qcut`).

In [11]:
ngay_moc = ht["ngay_dat"].max() + pd.Timedelta(days=1)
rfm = ht.groupby("ma_kh").agg(
    R=("ngay_dat", lambda x: (ngay_moc - x.max()).days),
    F=("ma_don", "nunique"),
    M=("doanh_thu", "sum"),
)
rfm["R_diem"] = pd.qcut(rfm["R"], 5, labels=[5, 4, 3, 2, 1]).astype(int)          # gần đây → điểm cao
rfm["F_diem"] = pd.qcut(rfm["F"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_diem"] = pd.qcut(rfm["M"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM"] = rfm["R_diem"].astype(str) + rfm["F_diem"].astype(str) + rfm["M_diem"].astype(str)
rfm.sort_values("M", ascending=False).head()

,R,F,M,R_diem,F_diem,M_diem,RFM
ma_kh,,,,,,,
KH0542,18,14,290436000.0,4,5,5,455
KH0475,29,16,239879000.0,4,5,5,455
KH0455,8,12,238994000.0,5,5,5,555
KH0513,15,11,233108000.0,5,5,5,555
KH0020,8,12,221724000.0,5,5,5,555


In [12]:
def phan_khuc(r):
    if r.R_diem >= 4 and r.F_diem >= 4:
        return "Khách hàng tốt nhất"
    if r.F_diem >= 4:
        return "Trung thành"
    if r.R_diem >= 4:
        return "Mới / tiềm năng"
    if r.R_diem <= 2 and r.F_diem >= 3:
        return "Có nguy cơ rời bỏ"
    if r.R_diem <= 2:
        return "Ngủ đông"
    return "Cần chăm sóc"

rfm["phan_khuc"] = rfm.apply(phan_khuc, axis=1)
tong_hop = rfm.groupby("phan_khuc").agg(so_kh=("R", "size"), R_tb=("R", "mean"), F_tb=("F", "mean"), M_tb_trieu=("M", lambda x: x.mean() / 1e6))
tong_hop.round(1).sort_values("M_tb_trieu", ascending=False)

,so_kh,R_tb,F_tb,M_tb_trieu
phan_khuc,,,,
Khách hàng tốt nhất,148,16.1,10.4,86.1
Trung thành,81,52.5,7.8,35.7
Có nguy cơ rời bỏ,61,206.3,3.6,9.2
Mới / tiềm năng,81,15.0,2.7,8.1
Cần chăm sóc,64,42.1,2.8,5.9
Ngủ đông,138,201.6,1.6,5.0


In [13]:
ht.to_csv("don_hang_sach.csv", index=False, encoding="utf-8-sig")
rfm.to_csv("rfm.csv", encoding="utf-8-sig")

## Phần B. Bài tập tự làm

Dùng các DataFrame `df` (đã làm sạch, gồm cả đơn hủy), `ht` (đơn hoàn thành) và `rfm` ở Phần A.

### Bài B1. Báo cáo chất lượng dữ liệu

Viết hàm `kiem_tra(df)` trả về DataFrame có một dòng cho mỗi cột với: kiểu dữ liệu, số giá trị thiếu, % thiếu,
số giá trị khác nhau. Chạy hàm trên `raw` và trên `df` để so sánh trước – sau khi làm sạch.

In [15]:
# Viết code của bạn ở đây
def kiem_tra(data):
    return pd.DataFrame({
        "kieu_du_lieu": data.dtypes,
        "so_thieu": data.isna().sum(),
        "phan_tram_thieu": (data.isna().mean() * 100).round(2),
        "so_gia_tri_khac_nhau": data.nunique()
    })

print("BÁO CÁO DỮ LIỆU THÔ (raw)")
display(kiem_tra(raw))

print("\nBÁO CÁO DỮ LIỆU SAU LÀM SẠCH (df)")
display(kiem_tra(df))

BÁO CÁO DỮ LIỆU THÔ (raw)


,kieu_du_lieu,so_thieu,phan_tram_thieu,so_gia_tri_khac_nhau
ma_don,object,0,0.00,3771
ngay_dat,object,0,0.00,511
ma_kh,object,0,0.00,600
vung,object,20,0.52,9
kenh,object,0,0.00,4
nhom_hang,object,0,0.00,6
san_pham,object,0,0.00,30
so_luong,float64,61,1.60,3
don_gia,object,0,0.00,1874
trang_thai,object,0,0.00,2



BÁO CÁO DỮ LIỆU SAU LÀM SẠCH (df)


,kieu_du_lieu,so_thieu,phan_tram_thieu,so_gia_tri_khac_nhau
ma_don,object,0,0.0,3761
ngay_dat,datetime64[ns],0,0.0,365
ma_kh,object,0,0.0,600
vung,object,0,0.0,3
kenh,object,0,0.0,4
nhom_hang,object,0,0.0,6
san_pham,object,0,0.0,30
so_luong,int64,0,0.0,3
don_gia,float64,0,0.0,1810
trang_thai,object,0,0.0,2


### Bài B2. Doanh thu theo vùng và quý

Từ `ht`, lập pivot table doanh thu (triệu đồng) với dòng là vùng, cột là quý. Thêm cột tăng trưởng quý 4 so với quý 3 (%).
Vùng nào tăng trưởng mạnh nhất?

In [16]:
# Viết code của bạn ở đây
pv_vung_quy = pd.pivot_table(
    ht, index="vung", columns="quy", values="doanh_thu", aggfunc="sum") / 1e6
pv_vung_quy["tang_truong_Q4_vs_Q3 (%)"] = (
    (pv_vung_quy[4] - pv_vung_quy[3]) / pv_vung_quy[3] * 100).round(2)
vung_max = pv_vung_quy["tang_truong_Q4_vs_Q3 (%)"].idxmax()
gia_tri_max = pv_vung_quy["tang_truong_Q4_vs_Q3 (%)"].max()

display(pv_vung_quy.round(2))
print(f"Vùng tăng trưởng mạnh nhất (Q4 so với Q3): {vung_max} (+{gia_tri_max}%)")

quy,1,2,3,4,tang_truong_Q4_vs_Q3 (%)
vung,,,,,
Miền Bắc,1646.43,1590.64,1257.27,1735.73,38.06
Miền Nam,1934.17,2173.14,1691.44,2020.70,19.47
Miền Trung,1186.90,717.66,760.65,1202.46,58.08


Vùng tăng trưởng mạnh nhất (Q4 so với Q3): Miền Trung (+58.08%)


### Bài B3. Tỷ lệ hủy đơn theo kênh

Từ `df` (gồm cả đơn hủy), tính tỷ lệ hủy đơn (%) theo từng kênh bán và theo nhóm hàng.
Kênh/nhóm hàng nào có tỷ lệ hủy cao nhất? Gợi ý một hành động cho quản lý.

<details><summary>💡 Gợi ý</summary>

Tạo cột 0/1 cho đơn hủy rồi lấy trung bình theo nhóm = tỷ lệ.

</details>

In [18]:
# Viết code của bạn ở đây
df["da_huy"] = (df["trang_thai"] == "Đã hủy").astype(int)
huy_theo_kenh = (df.groupby("kenh")["da_huy"].mean() * 100).round(2).sort_values(ascending=False)
huy_theo_nhom = (df.groupby("nhom_hang")["da_huy"].mean() * 100).round(2).sort_values(ascending=False)
pv_huy = (pd.pivot_table(df, index="nhom_hang", columns="kenh", values="da_huy", aggfunc="mean") * 100).round(2)

print("TỶ LỆ HỦY ĐƠN THEO KÊNH BÁN (%)")
display(huy_theo_kenh.to_frame(name="ty_le_huy (%)"))
print("\n TỶ LỆ HỦY ĐƠN THEO NHÓM HÀNG (%)")
display(huy_theo_nhom.to_frame(name="ty_le_huy (%)"))
print("\n CHI TIẾT TỶ LỆ HỦY THEO NHÓM HÀNG & KÊNH (%)")
display(pv_huy)
print(f"-> Kênh có tỷ lệ hủy cao nhất: {huy_theo_kenh.idxmax()} ({huy_theo_kenh.max()}%)")
print(f"-> Nhóm hàng có tỷ lệ hủy cao nhất: {huy_theo_nhom.idxmax()} ({huy_theo_nhom.max()}%)")

TỶ LỆ HỦY ĐƠN THEO KÊNH BÁN (%)


,ty_le_huy (%)
kenh,
App,21.12
Cửa hàng,19.44
Website,19.20
Sàn TMĐT,19.17



TỶ LỆ HỦY ĐƠN THEO NHÓM HÀNG (%)


,ty_le_huy (%)
nhom_hang,
Laptop,21.20
Điện thoại,20.87
Thời trang,19.55
Phụ kiện,19.34
Gia dụng,19.25
Mỹ phẩm,18.94



CHI TIẾT TỶ LỆ HỦY THEO NHÓM HÀNG & KÊNH (%)


kenh,App,Cửa hàng,Sàn TMĐT,Website
nhom_hang,,,,
Gia dụng,21.68,19.40,20.33,15.13
Laptop,23.93,18.06,18.88,22.13
Mỹ phẩm,15.64,14.10,23.30,20.45
Phụ kiện,18.11,22.58,18.53,20.00
Thời trang,22.33,22.68,15.61,19.07
Điện thoại,26.70,17.53,19.65,18.02


-> Kênh có tỷ lệ hủy cao nhất: App (21.12%)
-> Nhóm hàng có tỷ lệ hủy cao nhất: Laptop (21.2%)


### Bài B4. Top sản phẩm và giỏ hàng trung bình

(a) 10 sản phẩm có doanh thu cao nhất và tỷ trọng cộng dồn (%) của chúng (gợi ý `cumsum`).
(b) Giá trị đơn hàng trung bình (AOV) theo kênh và theo tháng; tháng nào AOV cao nhất?

In [19]:
# Viết code của bạn ở đây
tong_doanh_thu = ht["doanh_thu"].sum()
sp_doanh_thu = (
    ht.groupby("san_pham", as_index=False)["doanh_thu"]
    .sum()
    .sort_values(by="doanh_thu", ascending=False)
)
sp_doanh_thu["ty_trong (%)"] = (sp_doanh_thu["doanh_thu"] / tong_doanh_thu * 100).round(2)
sp_doanh_thu["ty_trong_cong_don (%)"] = sp_doanh_thu["ty_trong (%)"].cumsum().round(2)
sp_doanh_thu["doanh_thu_trieu"] = (sp_doanh_thu["doanh_thu"] / 1e6).round(2)

top10_sp = sp_doanh_thu.head(10)[["san_pham", "doanh_thu_trieu", "ty_trong (%)", "ty_trong_cong_don (%)"]]
print("TOP 10 SẢN PHẨM DOANH THU CAO NHẤT")
display(top10_sp)
don_hang = ht.groupby(["ma_don", "kenh", "thang"], as_index=False)["doanh_thu"].sum()
aov_kenh = (don_hang.groupby("kenh")["doanh_thu"].mean() / 1e6).round(2).sort_values(ascending=False)
aov_thang = (don_hang.groupby("thang")["doanh_thu"].mean() / 1e6).round(2)
pv_aov = (pd.pivot_table(don_hang, index="thang", columns="kenh", values="doanh_thu", aggfunc="mean") / 1e6).round(2)

print("\nGIÁ TRỊ ĐƠN HÀNG TRUNG BÌNH (AOV - TRIỆU ĐỒNG) THEO KÊNH")
display(aov_kenh.to_frame(name="AOV (triệu đồng)"))
print("\nGIÁ TRỊ ĐƠN HÀNG TRUNG BÌNH (AOV - TRIỆU ĐỒNG) THEO THÁNG")
display(aov_thang.to_frame(name="AOV (triệu đồng)"))
print("\nCHI TIẾT AOV THEO THÁNG & KÊNH (TRIỆU ĐỒNG)")
display(pv_aov)
print(f"-> Tháng có AOV cao nhất: Tháng {aov_thang.idxmax()} ({aov_thang.max()} triệu đồng/đơn)")

TOP 10 SẢN PHẨM DOANH THU CAO NHẤT


,san_pham,doanh_thu_trieu,ty_trong (%),ty_trong_cong_don (%)
7,Laptop C,2397.84,13.38,13.38
8,Laptop D,2177.40,12.15,25.53
6,Laptop B,2138.36,11.93,37.46
9,Laptop E,1997.87,11.15,48.61
5,Laptop A,1907.42,10.65,59.26
27,Điện thoại C,1237.93,6.91,66.17
28,Điện thoại D,1136.65,6.34,72.51
29,Điện thoại E,1069.86,5.97,78.48
26,Điện thoại B,1007.99,5.63,84.11
25,Điện thoại A,980.67,5.47,89.58



GIÁ TRỊ ĐƠN HÀNG TRUNG BÌNH (AOV - TRIỆU ĐỒNG) THEO KÊNH


,AOV (triệu đồng)
kenh,
Cửa hàng,6.45
Website,5.99
Sàn TMĐT,5.81
App,5.79



GIÁ TRỊ ĐƠN HÀNG TRUNG BÌNH (AOV - TRIỆU ĐỒNG) THEO THÁNG


,AOV (triệu đồng)
thang,
1,5.02
2,5.73
3,6.13
4,7.65
5,5.89
6,6.74
7,5.29
8,5.56
9,6.81



CHI TIẾT AOV THEO THÁNG & KÊNH (TRIỆU ĐỒNG)


kenh,App,Cửa hàng,Sàn TMĐT,Website
thang,,,,
1,4.18,7.01,5.43,4.24
2,5.46,7.26,5.77,5.25
3,6.68,5.86,5.50,6.42
4,7.98,6.48,6.30,9.52
5,7.37,8.18,4.96,4.14
6,5.19,6.56,8.11,7.11
7,6.81,4.50,6.82,2.50
8,6.17,4.60,3.04,8.02
9,4.90,5.99,6.12,10.92


-> Tháng có AOV cao nhất: Tháng 4 (7.65 triệu đồng/đơn)


### Bài B5. Nối dữ liệu RFM với thông tin khách hàng

Ghép `rfm` với `customers.csv` (dùng `merge`). Cho biết với mỗi phân khúc: số khách, tuổi trung bình, tỷ lệ nữ (%)
và vùng có nhiều khách nhất. Khách hàng có nguy cơ rời bỏ tập trung ở vùng nào?

In [23]:
rfm_df = rfm.reset_index() if "ma_kh" not in rfm.columns else rfm.copy()
cac_cot_pk = [c for c in rfm_df.columns if c not in ["ma_kh", "R", "F", "M", "R_diem", "F_diem", "M_diem", "RFM"]]

if cac_cot_pk:
    cot_pk = cac_cot_pk[0]  # Dùng chính cột phân khúc có sẵn từ phần A7
else:
    cot_pk = "phan_khuc"
    def phan_khuc_rfm(row):
        r, f, m = row["R_diem"], row["F_diem"], row["M_diem"]
        if r >= 4 and f >= 4 and m >= 4:
            return "Khách VIP"
        elif r <= 2 and f >= 3:
            return "Nguy cơ rời bỏ"
        elif r <= 2 and f <= 2:
            return "Đã ngủ quên"
        elif r >= 4 and f <= 2:
            return "Khách mới"
        else:
            return "Khách tiềm năng"
    rfm_df[cot_pk] = rfm_df.apply(phan_khuc_rfm, axis=1)
kh = pd.read_csv(DATA + "customers.csv")
rfm_kh = rfm_df.merge(kh, on="ma_kh", how="left")
rfm_kh["la_nu"] = (rfm_kh["gioi_tinh"] == "Nữ").astype(int)
bao_cao_pk = rfm_kh.groupby(cot_pk).agg(
    so_khach=("ma_kh", "count"),
    tuoi_tb=("tuoi", "mean"),
    ty_le_nu=("la_nu", lambda x: x.mean() * 100),
    vung_dong_nhat=("vung", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
).round({"tuoi_tb": 1, "ty_le_nu": 2})
print("BÁO CÁO PHÂN KHÚC KHÁCH HÀNG RFM")
display(bao_cao_pk)
ten_pk_nguy_co = [pk for pk in rfm_kh[cot_pk].dropna().unique() if "nguy cơ" in str(pk).lower() or "rời bỏ" in str(pk).lower()]
if ten_pk_nguy_co:
    nhom_nguy_co = ten_pk_nguy_co[0]
    phan_bo_vung = rfm_kh[rfm_kh[cot_pk] == nhom_nguy_co]["vung"].value_counts()
    print(f"\nPHÂN BỐ VÙNG CỦA NHÓM '{nhom_nguy_co}' ")
    display(phan_bo_vung.to_frame(name="so_khach"))
    if not phan_bo_vung.empty:
        print(f"-> Tập trung nhiều nhất ở vùng: {phan_bo_vung.idxmax()} ({phan_bo_vung.max()} khách)")
else:
    print("\nCác phân khúc hiện có trong bảng:", rfm_kh[cot_pk].unique())

BÁO CÁO PHÂN KHÚC KHÁCH HÀNG RFM


,so_khach,tuoi_tb,ty_le_nu,vung_dong_nhat
phan_khuc,,,,
Có nguy cơ rời bỏ,61,38.8,37.70,Miền Nam
Cần chăm sóc,64,42.0,56.25,Miền Bắc
Khách hàng tốt nhất,148,42.0,46.62,Miền Nam
Mới / tiềm năng,81,40.8,45.68,Miền Nam
Ngủ đông,138,42.2,52.90,Miền Bắc
Trung thành,81,41.7,56.79,Miền Nam



PHÂN BỐ VÙNG CỦA NHÓM 'Có nguy cơ rời bỏ' 


,so_khach
vung,
Miền Nam,27
Miền Bắc,20
Miền Trung,14


-> Tập trung nhiều nhất ở vùng: Miền Nam (27 khách)


## Phần C. Bài tập làm thêm

### Bài C1. Phân tích nhóm khách hàng theo tháng mua đầu tiên (cohort)

Xác định tháng mua đầu tiên của mỗi khách hàng. Lập bảng cohort: dòng = tháng mua đầu, cột = số tháng kể từ lần đầu
(0, 1, 2, ...), giá trị = tỷ lệ khách còn quay lại mua (%). Nhận xét tỷ lệ giữ chân sau 3 tháng.

In [27]:
# Viết code của bạn ở đây
cohort_df = ht[["ma_kh", "ngay_dat"]].copy()
cohort_df["thang_mua"] = cohort_df["ngay_dat"].dt.to_period("M")
cohort_df["thang_dau"] = cohort_df.groupby("ma_kh")["thang_mua"].transform("min")
cohort_df["khoang_thang"] = (cohort_df["thang_mua"] - cohort_df["thang_dau"]).apply(lambda x: x.n)
cohort_counts = (
    cohort_df.groupby(["thang_dau", "khoang_thang"])["ma_kh"]
    .nunique()
    .unstack(fill_value=0)
)
cohort_sizes = cohort_counts[0]
retention_matrix = cohort_counts.divide(cohort_sizes, axis=0) * 100
for i, thang in enumerate(retention_matrix.index):
    so_thang_toi_da = len(retention_matrix.index) - 1 - i
    retention_matrix.loc[thang, retention_matrix.columns > so_thang_toi_da] = np.nan

print(" BẢNG COHORT - TỶ LỆ KHÁCH QUAY LẠI MUA HÀNG (%) ")
display(retention_matrix.round(1))
tb_sau_3_thang = retention_matrix[3].mean().round(2)
print(f"\n-> Tỷ lệ giữ chân trung bình sau 3 tháng (Tháng thứ 3): {tb_sau_3_thang}%")
print(f"-> Nhóm giữ chân tốt nhất ở tháng 3: Cohort {retention_matrix[3].idxmax()} ({retention_matrix[3].max():.1f}%)")
print(f"-> Nhóm giữ chân thấp nhất ở tháng 3: Cohort {retention_matrix[3].idxmin()} ({retention_matrix[3].min():.1f}%)")

 BẢNG COHORT - TỶ LỆ KHÁCH QUAY LẠI MUA HÀNG (%) 


khoang_thang,0,1,2,3,4,5,6,7,8,9,10,11
thang_dau,,,,,,,,,,,,
2025-01,100.0,34.5,39.1,36.6,33.5,33.8,31.0,32.0,28.5,34.9,46.8,40.8
2025-02,100.0,45.2,35.5,45.2,37.1,32.3,38.7,38.7,40.3,43.5,56.5,NaN
2025-03,100.0,26.5,22.4,24.5,30.6,30.6,24.5,32.7,42.9,44.9,NaN,NaN
2025-04,100.0,21.4,32.1,21.4,21.4,21.4,32.1,28.6,28.6,NaN,NaN,NaN
2025-05,100.0,15.0,30.0,20.0,15.0,40.0,20.0,45.0,NaN,NaN,NaN,NaN
2025-06,100.0,22.2,0.0,16.7,11.1,38.9,22.2,NaN,NaN,NaN,NaN,NaN
2025-07,100.0,11.1,22.2,16.7,38.9,38.9,NaN,NaN,NaN,NaN,NaN,NaN
2025-08,100.0,35.7,28.6,28.6,42.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2025-09,100.0,22.2,66.7,11.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



-> Tỷ lệ giữ chân trung bình sau 3 tháng (Tháng thứ 3): 24.52%
-> Nhóm giữ chân tốt nhất ở tháng 3: Cohort 2025-02 (45.2%)
-> Nhóm giữ chân thấp nhất ở tháng 3: Cohort 2025-09 (11.1%)


### Bài C2. Doanh thu theo ngày trong tuần và trung bình trượt

(a) Doanh thu trung bình theo thứ trong tuần (sắp xếp Thứ 2 → Chủ nhật, đổi tên sang tiếng Việt).
(b) Tạo chuỗi doanh thu theo ngày (`resample("D")`), tính trung bình trượt 7 ngày và tìm 3 tuần có doanh thu cao nhất.

In [29]:
# Viết code của bạn ở đây
thu_map = {
    "Monday": "Thứ 2",
    "Tuesday": "Thứ 3",
    "Wednesday": "Thứ 4",
    "Thursday": "Thứ 5",
    "Friday": "Thứ 6",
    "Saturday": "Thứ 7",
    "Sunday": "Chủ nhật"
}
thu_tu_tuan = ["Thứ 2", "Thứ 3", "Thứ 4", "Thứ 5", "Thứ 6", "Thứ 7", "Chủ nhật"]
dt_theo_ngay = ht.groupby(["ngay_dat", "thu"], as_index=False)["doanh_thu"].sum()
dt_theo_ngay["thu_vn"] = dt_theo_ngay["thu"].map(thu_map)

dt_tb_thu = (
    dt_theo_ngay.groupby("thu_vn")["doanh_thu"]
    .mean()
    .reindex(thu_tu_tuan) / 1e6
).round(2)
print(" DOANH THU TRUNG BÌNH THEO THỨ TRONG TUẦN (TRIỆU ĐỒNG/NGÀY) ")
display(dt_tb_thu.to_frame(name="doanh_thu_tb (triệu đồng)"))
chuoi_ngay = (
    ht.set_index("ngay_dat")["doanh_thu"]
    .resample("D")
    .sum()
    .fillna(0)
)
df_ngay = pd.DataFrame({
    "doanh_thu_trieu": (chuoi_ngay / 1e6).round(2),
    "trung_binh_truot_7d_trieu": (chuoi_ngay.rolling(window=7).mean() / 1e6).round(2)
})
print("\n CHUỖI DOANH THU THEO NGÀY & TRUNG BÌNH TRƯỢT 7 NGÀY (10 NGÀY ĐẦU) ")
display(df_ngay.head(10))
doanh_thu_tuan = (chuoi_ngay.resample("W").sum() / 1e6).round(2)
top3_tuan = doanh_thu_tuan.sort_values(ascending=False).head(3)
print("\n TOP 3 TUẦN CÓ DOANH THU CAO NHẤT (TRIỆU ĐỒNG) ")
display(top3_tuan.to_frame(name="doanh_thu_tuan (triệu đồng)"))

 DOANH THU TRUNG BÌNH THEO THỨ TRONG TUẦN (TRIỆU ĐỒNG/NGÀY) 


,doanh_thu_tb (triệu đồng)
thu_vn,
Thứ 2,42.84
Thứ 3,42.92
Thứ 4,58.21
Thứ 5,51.77
Thứ 6,43.81
Thứ 7,58.40
Chủ nhật,45.49



 CHUỖI DOANH THU THEO NGÀY & TRUNG BÌNH TRƯỢT 7 NGÀY (10 NGÀY ĐẦU) 


,doanh_thu_trieu,trung_binh_truot_7d_trieu
ngay_dat,,
2025-01-01,53.93,NaN
2025-01-02,78.84,NaN
2025-01-03,63.27,NaN
2025-01-04,116.22,NaN
2025-01-05,93.67,NaN
2025-01-06,14.97,NaN
2025-01-07,108.92,75.69
2025-01-08,40.27,73.74
2025-01-09,99.18,76.64



 TOP 3 TUẦN CÓ DOANH THU CAO NHẤT (TRIỆU ĐỒNG) 


,doanh_thu_tuan (triệu đồng)
ngay_dat,
2025-01-12,581.22
2025-01-26,531.99
2025-12-14,513.49


### Bài C3. Đóng gói quy trình làm sạch

Viết hàm `lam_sach(duong_dan_raw, duong_dan_kh)` gom toàn bộ các bước A2–A6 thành một hàm, trả về DataFrame sạch.
Kiểm tra: kết quả của hàm phải có cùng số dòng với `df` ở Phần A. Vì sao đóng gói thành hàm lại quan trọng
khi dữ liệu được cập nhật hằng tuần?

In [30]:
# Viết code của bạn ở đây
def lam_sach(duong_dan_raw, duong_dan_kh):
    data = pd.read_csv(duong_dan_raw)
    kh = pd.read_csv(duong_dan_kh)
    data = data.drop_duplicates().copy()
    data["vung"] = data["vung"].str.strip().str.title()
    def lam_sach_gia(x):
        if isinstance(x, str):
            x = x.replace("đ", "").replace(".", "").strip()
        return float(x)

    data["don_gia"] = data["don_gia"].apply(lam_sach_gia)
    data["so_luong"] = pd.to_numeric(data["so_luong"], errors="coerce")
    data["ngay_dat"] = pd.to_datetime(data["ngay_dat"], format="mixed", dayfirst=True)
    data["thang"] = data["ngay_dat"].dt.month
    data["quy"] = data["ngay_dat"].dt.quarter
    data["thu"] = data["ngay_dat"].dt.day_name()
    data["so_luong"] = data["so_luong"].fillna(data["so_luong"].median())
    vung_kh = kh.set_index("ma_kh")["vung"]
    data["vung"] = data["vung"].fillna(data["ma_kh"].map(vung_kh))
    data = data[data["don_gia"] > 0]
    def la_ngoai_lai(g):
        q1, q3 = g.quantile([0.25, 0.75])
        iqr = q3 - q1
        return (g < q1 - 1.5 * iqr) | (g > q3 + 1.5 * iqr)

    ngoai_lai = data.groupby("nhom_hang")["don_gia"].transform(la_ngoai_lai)
    data = data[~ngoai_lai].copy()
    data["so_luong"] = data["so_luong"].astype(int)
    data["doanh_thu"] = data["so_luong"] * data["don_gia"]

    return data
df_sach = lam_sach(DATA + "ecommerce_orders_raw.csv", DATA + "customers.csv")

print(f"Số dòng của df ở Phần A : {len(df)}")
print(f"Số dòng của hàm lam_sach: {len(df_sach)}")
print("Khớp hoàn toàn số dòng? :", len(df) == len(df_sach))
display(df_sach.head())

Số dòng của df ở Phần A : 3761
Số dòng của hàm lam_sach: 3761
Khớp hoàn toàn số dòng? : True


,ma_don,ngay_dat,ma_kh,vung,kenh,nhom_hang,san_pham,so_luong,don_gia,trang_thai,thang,quy,thu,doanh_thu
0,DH002986,2025-01-24,KH0482,Miền Nam,Website,Phụ kiện,Phụ kiện D,2,371000.0,Hoàn thành,1,1,Friday,742000.0
1,DH002027,2025-12-18,KH0311,Miền Nam,Website,Phụ kiện,Phụ kiện C,3,450000.0,Hoàn thành,12,4,Thursday,1350000.0
2,DH001811,2025-06-05,KH0280,Miền Trung,Cửa hàng,Laptop,Laptop C,1,16332000.0,Hoàn thành,6,2,Thursday,16332000.0
3,DH000608,2025-03-06,KH0092,Miền Trung,App,Phụ kiện,Phụ kiện C,3,503000.0,Hoàn thành,3,1,Thursday,1509000.0
4,DH002808,2025-01-26,KH0446,Miền Bắc,Sàn TMĐT,Mỹ phẩm,Mỹ phẩm E,2,421000.0,Hoàn thành,1,1,Sunday,842000.0
